# 🟦 1. **Introduction**
## E-Commerce Transactions Data Quality & Revenue Analysis

### Objective
Clean messy transaction data and generate reliable revenue insights.

#### Business Question
How can we ensure accurate revenue reporting and uncover key business drivers?

# 🟦 2. Load Data

In [64]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/ecommerce_transactions_dirty.csv")

df.head()
df.shape
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1530 entries, 0 to 1529
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   order_id          1530 non-null   int64  
 1   order_date        1530 non-null   object 
 2   customer_id       1530 non-null   int64  
 3   customer_age      1381 non-null   float64
 4   gender            1221 non-null   object 
 5   country           1530 non-null   object 
 6   product_category  1530 non-null   object 
 7   unit_price        1478 non-null   float64
 8   quantity          1530 non-null   int64  
 9   discount          1126 non-null   float64
 10  payment_method    1530 non-null   object 
dtypes: float64(3), int64(3), object(5)
memory usage: 131.6+ KB


# 🟦 3. Data Quality Assessment

In [42]:
# Missing values
df.isnull().sum()

# Duplicate check
df.duplicated(subset='order_id').sum()

np.int64(30)

### Key Issues Identified
- Missing values in unit_price, discount, customer_age
- Duplicate order_ids
- Incorrect data types
- Inconsistent text fields

# 🟦 4. Data Cleaning

In [43]:
# Remove duplicates
df = df.drop_duplicates(subset='order_id')

# Fix types
df['unit_price'] = pd.to_numeric(df['unit_price'], errors='coerce')
df['discount'] = pd.to_numeric(df['discount'], errors='coerce')

# Fill missing values
df['discount'] = df['discount'].fillna(0)
df['customer_age'] = df['customer_age'].fillna(df['customer_age'].median())

# Fill unit_price (by product median)
df['unit_price'] = df.groupby('product_category')['unit_price'].transform(lambda x: x.fillna(x.median()))

# Standardize text
df['gender'] = df['gender'].str.strip().str.lower().replace({'m':'male','f':'female'})
df['gender'] = df['gender'].fillna('unknown')

df['country'] = df['country'].str.strip().str.title().fillna('Unknown')
df['product_category'] = df['product_category'].str.strip().str.title()

# Convert date
df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')

### Cleaning Decisions
- Dropped duplicate orders to prevent revenue inflation
- Filled missing discount with 0 (assumed no discount)
- Used median imputation for unit_price and customer_age
- Standardized categorical fields for consistency

🟦 5. Feature Engineering

# 🟦 5. Feature Engineering

In [44]:
df['revenue'] = df['unit_price'] * df['quantity'] * (1 - df['discount'])

df['order_month'] = df['order_date'].dt.to_period('M')

df['age_group'] = pd.cut(
    df['customer_age'],
    bins=[0,18,25,35,45,55,65,100],
    labels=['<18','18-24','25-34','35-44','45-54','55-64','65+']
)

df['is_discounted'] = np.where(df['discount'] > 0, 'Yes', 'No')


# 🟦 6. Revenue & Business Performance Analysis

In [50]:
# Total revenue
df['revenue'].sum()

# Monthly revenue
monthly_revenue = df.groupby('order_month')['revenue'].sum().reset_index()

# Customer LTV
customer_ltv = df.groupby('customer_id')['revenue'].sum().reset_index()

# Discount split
discount_revenue_split = df.groupby('is_discounted')['revenue'].sum().reset_index()



# 🟦 7. Time Series & Customer Behavior Analysis

In [51]:
# MoM growth
monthly_revenue = monthly_revenue.sort_values('order_month')
monthly_revenue['mom_growth'] = monthly_revenue['revenue'].pct_change()

# Rolling averages
daily_rev = df.groupby('order_date')['revenue'].sum().reset_index()
daily_rev['rolling_7'] = daily_rev['revenue'].rolling(7).mean()
daily_rev['rolling_30'] = daily_rev['revenue'].rolling(30).mean()

# 🟦 8. Risk Assessment & Anomaly Detection

In [52]:
# Revenue concentration
customer_ltv = customer_ltv.sort_values('revenue', ascending=False)
customer_ltv['cum_pct'] = customer_ltv['revenue'].cumsum() / customer_ltv['revenue'].sum()

# Discount risk
df.groupby('is_discounted')['revenue'].sum()

is_discounted
No     504462.780
Yes    444710.074
Name: revenue, dtype: float64

# 🟦 9. Data Validation & Quality Checks

In [53]:
(df['revenue'] < 0).sum()
(df['discount'] > 1).sum()

df[['order_id','unit_price','quantity']].isnull().sum()

order_id      0
unit_price    0
quantity      0
dtype: int64

# 🟦 10. Export Clean Data & Tables

In [65]:
import os

# Create folders if they don't exist
os.makedirs("../data/cleaned", exist_ok=True)
os.makedirs("../outputs", exist_ok=True)

df.to_csv("../data/cleaned/ecommerce_transactions_clean.csv", index=False)

monthly_revenue.to_csv("../outputs/monthly_revenue.csv", index=False)
customer_ltv.to_csv("../outputs/customer_ltv.csv", index=False)
discount_revenue_split.to_csv("../outputs/discount_revenue_split.csv", index=False)